# Nepali expressive TTS — Kaggle GPU
Before **Run all**: enable a GPU accelerator, accept access at https://huggingface.co/ai4bharat/indic-parler-tts, and add a private Kaggle Secret named `HF_TOKEN` containing a token from the approved Hugging Face account.

In [ ]:
!pip install -q "transformers>=4.46,<5" "accelerate>=1.0" "soundfile>=0.12" "git+https://github.com/huggingface/parler-tts.git"
!rm -rf /kaggle/working/tts-research
!git clone -q https://github.com/BlazeWild/tts-research.git /kaggle/working/tts-research

In [ ]:
import os, sys, torch
from kaggle_secrets import UserSecretsClient
os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['TTS_DEVICE'] = 'cuda'
os.environ['TTS_DTYPE'] = 'float16'
sys.path.insert(0, '/kaggle/working/tts-research')
assert torch.cuda.is_available(), 'GPU is not enabled: Notebook options → Accelerator → GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import soundfile as sf
from IPython.display import Audio, display
from tts_engine import IndicParlerEngine
from voice_styles import make_description

engine = IndicParlerEngine()
output_dir = Path('/kaggle/working/nepali_tts_outputs')
output_dir.mkdir(exist_ok=True)
samples = [
    ('welcome', 'नमस्कार! नागरिक सेवामा तपाईंलाई स्वागत छ। म तपाईंलाई कसरी सहयोग गर्न सक्छु?', 'Warm conversation', 'Normal', 'Natural'),
    ('news', 'आज देशका अधिकांश स्थानमा मौसम सफा रहने अनुमान गरिएको छ।', 'News', 'Normal', 'Natural'),
    ('happy', 'वाह! तपाईंको आवेदन सफलतापूर्वक स्वीकृत भयो!', 'Happy', 'Fast', 'High'),
]

for name, text, emotion, speed, pitch in samples:
    result = engine.synthesize(text, make_description(emotion, speed, pitch))
    path = output_dir / f'{name}.wav'
    sf.write(path, result.audio, result.sample_rate)
    print(f'{path.name}: {len(result.audio)/result.sample_rate:.1f}s, generated in {result.elapsed_seconds:.1f}s on {result.device}')
    display(Audio(filename=str(path)))

In [ ]:
# Edit this text and rerun this cell for your own answer.
my_text = 'तपाईंको प्रश्नका लागि धन्यवाद। म आवश्यक जानकारी खोजेर स्पष्ट जवाफ दिनेछु।'
result = engine.synthesize(my_text, make_description('Warm conversation', 'Normal', 'Natural'))
my_path = output_dir / 'my_answer.wav'
sf.write(my_path, result.audio, result.sample_rate)
display(Audio(filename=str(my_path)))
print('Saved:', my_path)

In [ ]:
!cd /kaggle/working && zip -qr nepali_tts_outputs.zip nepali_tts_outputs
print('Download /kaggle/working/nepali_tts_outputs.zip from the notebook Files panel.')